### setup

In [1]:
# !pip install -U sentence-transformers
# !pip install datasets

In [39]:
!pip install tensorflow_ranking

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
     |████████████████████████████████| 151 kB 1.1 MB/s eta 0:00:01
     |████████████████████████████████| 17.3 MB 7.6 MB/s eta 0:00:01
     |████████████████████████████████| 479.6 MB 46.8 MB/s eta 0:00:01
     |████████████████████████████████| 294 kB 50.0 MB/s eta 0:00:01
     |████████████████████████████████| 5.6 MB 22.1 MB/s eta 0:00:01
     |████████████████████████████████| 17.3 MB 22.2 MB/s eta 0:00:01�████████▊           | 11.2 MB 22.2 MB/s eta 0:00:01
     |████████████████████████████████| 1.7 MB 22.2 MB/s eta 0:00:01
     |████████████████████████████████| 2.4 MB 30.3 MB/s eta 0:00:01
     |████████████████████████████████| 440 kB 30.6 MB/s eta 0:00:01
     |████████████████████████████████| 22.9 MB 28.5 MB/s eta 0:00:01
     |████████████████████████████████| 133 kB 30.3 MB/s eta 0:00:01
     |████████████████████████████████| 5.4 MB 30.4 MB/s eta 0:00:01
     |████████████████████████████████| 186 

  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.3.2
    Uninstalling google-auth-2.3.2:
      Successfully uninstalled google-auth-2.3.2
  Attempting uninstall: tensorboard-data-server
    Found existing installation: tensorboard-data-server 0.6.1
    Uninstalling tensorboard-data-server-0.6.1:
      Successfully uninstalled tensorboard-data-server-0.6.1
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.19.0
    Uninstalling protobuf-3.19.0:
      Successfully uninstalled protobuf-3.19.0
  Attempting uninstall: numpy
    Found existing installation: numpy 1.19.5
    Uninstalling numpy-1.19.5:
      Successfully uninstalled numpy-1.19.5
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.41.1
    Uninstalling grpcio-1.41.1:
      Successfully uninstalled grpcio-1.41.1
  Attempting uninstall: google-auth-oauthlib
    Found existing installation: google-auth-oauthlib 0.4.6
    Uninstalling google-auth-oauth

In [1]:
import pickle
import numpy as np
import os
import tqdm
import torch
import gc

from tensorflow.python.ops.numpy_ops import np_config
np_config.enable_numpy_behavior()

from importlib import reload
from matplotlib import pyplot as plt

import matrix_approx_zeshel

matrix_approx_zeshel = reload(matrix_approx_zeshel)

!ls  | grep matrix_approx_zeshel

2024-01-25 01:12:32.175966: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-01-25 01:12:32.281015: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-01-25 01:12:32.282382: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-01-25 01:12:33.558275: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


matrix_approx_zeshel.py


In [2]:
import collections
import pickle
import numpy as np
import tqdm
import os
import gc

import matplotlib.pyplot as plt


def load(limit, raw_path = "stand/log.local.logtime2.txt", path = "log.local.logtime2.bin", key_games = None, seed = 17, det_attempts = 0):
    readvector = lambda s : np.array(list(map(float, s.strip()[1:-2].split(","))))
    requests = list()
    docembs = collections.defaultdict(dict)

    if os.path.isfile(path):
        with open(path, "rb") as f:
            flimit, frequests, fdocembs = pickle.load(f)
            if flimit == limit:
                requests, docembs = frequests, fdocembs
            else:
                print(f"WARN: buffered limit is different, {flimit} != {limit}, reloading...")

    if not requests:
        with open(raw_path) as f:
            req = list()
            reqid = None
            models = list()
            prevreqmodel = None
            reqmodel = dict()
            prevmodelid = -1
            bannermodelid = -1
            for i, line in tqdm.tqdm_notebook(enumerate(f)):
                if line.startswith("Model = 6;"):
                    prevreqmodel = reqmodel
                    reqmodel = dict()

                if line.startswith("Model = "):
                    spl = line.split(" ")
                    prevmodelid = int(spl[2][:-1])
                    bannermodelid = max(bannermodelid , prevmodelid)
                    reqmodel[prevmodelid] = readvector(spl[3])
                elif line.startswith("dbid"):
                    spl = line.split(" ")
                    dbid = int(spl[1][:-1])
                    docembs[bannermodelid][dbid] = readvector(spl[2])
                elif line.startswith("seed"):
                    if len(requests) >= limit:
                        break
                    if req:
                        requests.append((reqid, prevreqmodel, sorted(req)))
                        req = list()
                    reqid = "$_" + (line.split()[1] + "_" + line.split()[3])
                else:
                    req.append(
                        (int(line.split()[0]), float(line.split()[1]))
                    )
        
        with open(path, "wb") as f:
            pickle.dump((limit, requests, docembs), f)

    games_count = len(requests[0][2])
    assert games_count == 16514
    requests = [r for r in requests if len(r[2]) == games_count]
    
    print([(i, len(docembs[i].keys())) for i in docembs])  # should be equal
    docblocks = {
        mid : np.array([x[1] for x in sorted(list(docembs[mid].items()))])
        for mid in docembs
    }
    
    class EvalContext:
        def __init__(self, games_count = games_count, key_size = 100, train_size = 0.7, key_games = None, seed = 17, det_attempts = 0):
            self.games_count = games_count
            
            self.key_size = key_size
            self.key_games = (
                np.random.choice(np.arange(games_count), key_size, replace=False)
                if key_games is None else
                key_games
            )

            self.requests = requests
            np.random.seed(seed)
            np.random.shuffle(self.requests)
            
            self.try_det_attempts(det_attempts)

            self.key_reqs = self.requests[:key_size + 1]
            self.key_reqs_idx = np.arange(key_size + 1)

            self.train_split = int(len(self.requests) * train_size)

            assert key_size + 1 < self.train_split

            self.train_reqs = self.requests[key_size + 1: self.train_split]
            self.test_reqs = self.requests[self.train_split:]

            self.slices = ["key", "train", "test"]
            print(len(self.key_reqs), len(self.train_reqs), len(self.test_reqs))

            self.docblocks = docblocks
            self.relevs = dict()
            
        def get_top_games(self):
            if not hasattr(self, "top_games"):
                embed_games = np.array([
                    np.array([r[2][g_i][1] for r in self.get_requests("train")])
                    for g_i in range(self.games_count)
                ])

                self.embed_games_mean = embed_games.mean(axis=1)
                self.top_games_all = (-self.embed_games_mean).argsort()
                self.top_games = self.top_games_all[:len(self.key_games)]

            return self.top_games
        
        def set_top_games_as_key(self):
            self.key_games = self.get_top_games()
            return self
        
        def get_kmeans_games(self, all_from_labels=True):
            X = self.get_relevs("train").T

            k_func = (
                (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
                if not all_from_labels else
                (lambda C : from_labels(X, C.labels_))
            )
            K_KMeans = k_func(
                KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
            )

            return K_KMeans

        def set_kmeans_games_as_key(self, *args, **kwargs):
            self.key_games = self.get_kmeans_games(*args, **kwargs)
            return self

        def try_det_attempts(self, det_attempts, model_id = 6):
            def get_det(r, r_i_array):
                kr = np.array([
                    r[r_i][1][model_id]
                    for r_i in r_i_array[:100]
                ])
                return np.abs(np.linalg.det(kr[:kr.shape[1], :]))

            best_i_array = np.arange(len(self.requests))

            for _ in range(det_attempts):
                # print("try update key_reqs...")
                
                r_i_array = np.arange(len(self.requests))
                np.random.shuffle(r_i_array)
                
                n, o = get_det(self.requests, r_i_array), get_det(self.requests, best_i_array)
                # print(n, o)
                if n > o:
                    best_i_array = r_i_array
                    # print("updated!")

            print("Best det = ", get_det(self.requests, best_i_array))
            
            new_requests = [
                self.requests[i]
                for i in best_i_array
            ]
            
            del self.requests
            gc.collect()

            self.requests = new_requests
            print(get_det(self.requests, np.arange(len(self.requests))))

        def get_relevs(self, t = "train"):
            if t not in self.relevs:
                self.relevs[t] = np.array([
                    np.array([g_i[1] for g_i in r[2]])
                    for r in self.get_requests(t)
                ])
                
            return self.relevs[t]

        def get_requests(self, t = "train"):
            if t == "train":
                return self.train_reqs
            elif t == "key":
                return self.key_reqs
            elif t == "test":
                return self.test_reqs
            else:
                assert False

    return EvalContext(key_games = key_games, seed = seed, det_attempts = det_attempts)

In [3]:
from sklearn.cluster import KMeans
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances


def from_labels(X, labels):
    K = list()
    for i in range(100):
        sl_i = np.arange(X.shape[0])[labels == i]
        sl = X[labels == i]
        
        if len(sl_i) == 0:
            # bug-mode
            while True:
                chosen = np.random.choice(np.arange(X.shape[0])[labels < i], size=1)[0]
                if chosen not in K:
                    K.append(chosen)
                    break
            continue
            
        center = sl.mean(axis=0).reshape(1, -1)
        best = euclidean_distances(sl, center).argmin()
        K.append(sl_i[best])
        assert labels[K[-1]] == i
    return K

class EvalContextRelevs:
    def __init__(self, relevs, key_size = 100, train_size = 0.7, key_games = None, seed = 17, shuffle=False, det_attempts = 0):
        self.relevs = np.array(relevs)
        self.reqs_count = self.relevs.shape[0]
        self.games_count = self.relevs.shape[1]
        
        self.key_size = key_size
        self.key_games = (
            np.random.choice(np.arange(self.games_count), key_size, replace=False)
            if key_games is None else
            key_games
        )
        np.random.seed(seed)
        if shuffle:
            np.random.shuffle(self.relevs)

        self.try_det_attempts(det_attempts)
        self.train_split = int(self.reqs_count * train_size)

        assert key_size + 1 < self.train_split


        self.key_relevs = self.relevs[:key_size]
        self.train_relevs = self.relevs[key_size + 1: self.train_split]
        self.test_relevs = self.relevs[self.train_split:]

        self.slices = ["key", "train", "test"]
        print(len(self.key_relevs), len(self.train_relevs), len(self.test_relevs))

    def get_top_games(self):
        return self.relevs.mean(axis=0).argsort()[:100]

    def set_top_games_as_key(self):
        self.key_games = self.get_top_games()
        return self

    def get_kmeans_games(self, all_from_labels=True):
        X = self.get_relevs("train").T

        k_func = (
            (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
            if not all_from_labels else
            (lambda C : from_labels(X, C.labels_))
        )
        K_KMeans = k_func(
            KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
        )
        
        return K_KMeans
    
    def set_kmeans_games_as_key(self, *args, **kwargs):
        self.key_games = self.get_kmeans_games(*args, **kwargs)
        return self
    

    def try_det_attempts(self, det_attempts):
        def get_det(r):
            kr = r[:self.key_size, self.key_games] - r.mean()
            return np.abs(np.linalg.det(kr))

        best_i_array = np.arange(len(self.relevs))

        for i in range(det_attempts):

            r_i_array = np.arange(len(self.relevs))
            np.random.shuffle(r_i_array)

            n, o = get_det(self.relevs[r_i_array, :]), get_det(self.relevs[best_i_array, :])
            
            # print(f"try update key_reqs ({o} vs {n}...")
            if n > o:
                best_i_array = r_i_array
                print(f"updated det ({i}, {o} -> {n})")

        print("Best det = ", get_det(self.relevs[best_i_array, :]))

        self.relevs = self.relevs[best_i_array, :]
        print("Current de = ", get_det(self.relevs))
        
    def get_relevs(self, t = "train"):
        if t == "train":
            return self.train_relevs
        elif t == "key":
            return self.key_relevs
        elif t == "test":
            return self.test_relevs
        else:
            assert False
            
    def get_requests(self, t = "train"):
        if t == "train":
            return self.train_reqs
        elif t == "key":
            return self.key_reqs
        elif t == "test":
            return self.test_reqs
        else:
            assert False

In [4]:
import tensorflow as tf
import copy
import tensorflow_ranking as tfr

class Popular:
    def __init__(self, ctx):
        self.ctx = ctx
        self.game_avg_scores = {
            t : self.get_user_scores(t).mean(axis = 0)
            for t in self.ctx.slices
        }
        self.trus_top = dict()
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        return [self.top_logits for _ in self.get_user_scores(t)]
    
    def get_score(self, t, topsize = 100):    
        recs = self.recommend(t)
        
        if isinstance(recs, list):
            recs = np.array(recs)
        
        mse = np.mean((recs - self.get_user_scores(t)) ** 2)
        
        if isinstance(recs, tf.Tensor):
            recs = tf.argsort(-recs, axis=1)[:, :topsize].numpy()
        else:
            recs = np.argsort(-recs, axis=1)[:, :topsize]
        
        if (t, topsize) not in self.trus_top:
            trus = self.get_user_scores(t)
            trus = np.argsort(-trus, axis=1)[:, :topsize]
            self.trus_top[(t, topsize)] = trus
            
        trus = self.trus_top[(t, topsize)]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)
    
import tensorflow as tf
import copy
import tensorflow_ranking as tfr

DEFAULT_FIT_KWARGS = {
    "learning_rate": 0.001,
    "n": 500,
    "c": 5000,
    "matrix_l2": 0,
    "dssm_l2": 0,
    "train_popbias": False,
    "train_bias": False,
    "train_vbias": False,
    "verbose": True,
    "train_matrix": True,
    "train_dssm": False,
    "loss": "mse",
    "ubatch": 1e9,
    "activation": "relu",
    "score_verbose": 0,
    "trainable_items": False,
    "use_keys_in_train": False,
    "Winit": "glorot0.01",
    "Wfreeze": False,
    "TEinit": "legacy",
    "normalize_embs": True
}

class CBKnnV0(Popular):
    def __init__(self, ctx, fit_kwargs = dict()):
        super().__init__(ctx)
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(fit_kwargs)
        self.fit_kwargs = p
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        
        self.embed_users = {
            t : np.array([
                    np.array([r_i[i] for i in sorted(ctx.key_games)])
                    for r_i in ctx.get_relevs(t)
                ])
            for t in ctx.slices
        }
        self.embed_users_mean = {
            t : self.embed_users[t].mean(axis = 0)
            for t in ctx.slices
        }
        # embed_users = embed_users - embed_users_mean
        print("self.embed_users['train'].shape = ", self.embed_users['train'].shape )
        
        self.embed_games = np.array([
            np.array([r[g_i] for r in ctx.get_relevs("key")])
            for g_i in range(ctx.games_count)
        ])
        
        self.embed_games_mean = self.embed_games.mean(axis = 0)
        
        # embed_games = embed_games - embed_games_mean
        print("self.embed_games.shape = ", self.embed_games.shape)
        
        
        if self.fit_kwargs["trainable_items"]:
            if self.fit_kwargs["TEinit"] == "legacy":
                value = self.embed_games
            elif self.fit_kwargs["TEinit"] == "anncur":
                key_cols_idx = np.array(sorted(ctx.key_games))
                value = (
                    np.linalg.pinv(ctx.get_relevs("train")[:, key_cols_idx])
                    @ ctx.get_relevs("train")
                ).T
                print(type(value))
            else:
                assert False, "unknown TEinit: " + str(self.fit_kwargs["TEinit"])
                
            self.trainable_games = tf.Variable(
                tf.convert_to_tensor(value, dtype=tf.float32),
                trainable=True
            )
        else:
            if self.fit_kwargs["TEinit"] == "anncur":
                key_cols_idx = np.array(sorted(ctx.key_games))
                self.embed_games = (
                    np.linalg.pinv(ctx.get_relevs("train")[:, key_cols_idx])
                    @ ctx.get_relevs("train")
                ).T
                print("ANNCur init")
            
        
        self.games_top_key = self.embed_games.mean(axis = 1)
        
        self.games2users = np.array([
            self.embed_games[g_i]
            for g_i in ctx.key_games
        ])
        print("self.games2users.shape = ", self.games2users.shape)
        
        self.core_users_scores = ctx.get_relevs("key")
        print("self.core_users_scores.shape = ", self.core_users_scores.shape)
        
        self.core_users_embs = self.embed_users["key"]
        print("self.core_users_embs.shape = ", self.core_users_embs.shape)
        
        self.ge_users = (
            (self.embed_users["train"].T / self.embed_users["train"].mean(axis=1)).T @ self.games2users
        )
        # ge_users = embed_users @ games2users
        print("self.ge_users.shape = ", self.ge_users.shape)
    
    def __repr__(self):
        return "CbKnn(" + str(self.fit_kwargs) + ")"
        
    # inherited   
    # def get_user_scores(self, t):
    
    def get_user_embs(self, t):
        if self.fit_kwargs["normalize_embs"]:
            return (self.embed_users[t] - self.embed_users_mean["train"]) / self.embed_users[t].std(axis=0)
        else:
            return self.embed_users[t]
    
    def get_game_embs(self):
        if not self.fit_kwargs["trainable_items"]:
            if self.fit_kwargs["normalize_embs"]:
                return (self.embed_games - self.embed_games.mean(axis=0)) / self.embed_games.std(axis=0)
            else:
                return self.embed_games 
        else:
            return self.trainable_games

    def fit(self, **kwargs):
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        p.update(kwargs)

        self.p = p
        self.train_bias = p["train_bias"]
        self.train_vbias = p["train_vbias"]
        self.train_popbias = p["train_popbias"]
        self.train_matrix = p["train_matrix"]
        self.train_dssm = p["train_dssm"]
        self.loss = p["loss"]

        if p["use_keys_in_train"]:
            train_user_scores = np.vstack([
                self.get_user_scores("key"),
                self.get_user_scores("train")
            ])
            train_user_embs = np.vstack([
                self.get_user_embs("key"),
                self.get_user_embs("train")
            ])
        else:
            train_user_scores = self.get_user_scores("train")
            train_user_embs = self.get_user_embs("train")

        game_embs = self.get_game_embs()
        
        
        
        if self.p["Winit"] == "glorot0.01":
            initializer = tf.keras.initializers.GlorotUniform()
            values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
            values = values / 100.  
        elif self.p["Winit"] == "glorot":
            initializer = tf.keras.initializers.GlorotUniform()
            values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
        elif self.p["Winit"] == "eye":
            values = np.eye(train_user_embs.shape[1], game_embs.shape[1])
        else:
            assert False, "init is not known:" + str(self.p["init"])
            
        self.W = tf.Variable(values, trainable = True) 
        self.pb = tf.Variable(1., trainable = True) 
        self.b = tf.Variable(0., trainable = True) 
        
        if p["verbose"]:
            print("self.W = ", self.W)
            print("0-loss = ", tf.reduce_mean((train_user_scores - 0) ** 2))
    
        opt =  tf.keras.optimizers.Adam(learning_rate=p["learning_rate"])

        if self.train_dssm:
            self.train_dssm = True

            dim = game_embs.shape[1]
            self.g_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.g_dssm(game_embs)
            
            # dim = train_user_embs.shape[1]
            self.u_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.u_dssm(train_user_embs)
            
        if self.train_vbias:
            self.vb = tf.Variable(
                np.zeros_like(self.game_avg_scores["train"], dtype=np.float32),
                trainable = True
            ) 
        
        
        for i in range(p["n"]):
            def loss():
                def get_logits_scores(loss_batch = 1e9):
                    game_slice = None
                    
                    ubatch = p["ubatch"]
                    if ubatch >= train_user_scores.shape[0]:
                        train_user_embs_ = train_user_embs
                        scores_ = train_user_scores
                    else:
                        u_slice = np.random.choice(np.arange(train_user_scores.shape[0]), ubatch, replace = True)
                        train_user_embs_ = train_user_embs[u_slice]
                        scores_ = train_user_scores[u_slice]
                    
                    if loss_batch >= game_embs.shape[0]:
                        game_embs_ = game_embs
                        scores_ = scores_
                    else:
                        game_slice = np.random.choice(np.arange(game_embs.shape[0]), loss_batch, replace = True)
                        game_embs_ = (
                            game_embs[game_slice]
                            if isinstance(game_embs, np.ndarray) else
                            tf.gather(game_embs, game_slice)
                        )
                        scores_ = scores_[:, game_slice]
                        
                    
                    logits = 0
                    
                    if self.train_matrix:
                        logits += train_user_embs_ @ self.W @ tf.transpose(game_embs_)

                    if self.train_dssm:
                        logits += self.u_dssm(train_user_embs_) @ tf.transpose(self.g_dssm(game_embs_))
                    
                    if self.train_popbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.pb * self.game_avg_scores["train"]
                        else:
                            logits += self.pb * self.game_avg_scores["train"][game_slice]
                        
                    if self.train_vbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.vb
                        else:
                            logits += tf.gather(self.vb, game_slice)
                        
                    if self.train_bias:
                        logits += self.b
                        
                    return logits, scores_
                        
                if self.loss == "mse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    v = tf.reduce_mean((scores - logits) ** 2)
                elif self.loss == "qmse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    q_mean = scores.mean(axis=1)
                    v = tf.reduce_mean(((scores.T - q_mean).T - logits) ** 2)
                elif self.loss == "ApproxNDCGLoss":
                    while True:
                        logits, scores = get_logits_scores(
                            loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))

                        v = tfr.keras.losses.ApproxNDCGLoss()(
                            scores.astype(np.float32),
                            logits
                        )
                    
                        if not tf.math.is_nan(v).numpy().any():
                            break
                        else:
                            if p["verbose"]:
                                print("nanloss ignored")
                elif self.loss == "softmax":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ))
                elif self.loss == "softmax_signed":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    sf = tf.nn.softmax(logits, axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        sf,
                        -sf
                    ))
                elif self.loss == "softmax_weighted":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ) * scores)
                elif self.loss == "sigmoid_top_100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    mask = np.argsort(-scores, axis=1) < 100
                    v = -tf.reduce_sum(
                        tf.math.log_sigmoid((2 * mask - 1) * logits)
                    )
                elif self.loss == "KL100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    mask = np.argsort(-scores, axis=1) < 100
                    v = tf.keras.losses.KLDivergence()((scores.T >= qs.T).T, logits)
                else:
                    assert False
                    
                if self.p["c"]:
                    v += tf.reduce_mean(self.W * self.W) * p["c"]
                    
                if self.p["dssm_l2"]:
                    for weights_ in [self.u_dssm.weights, self.g_dssm.weights]:
                        for weight_ in weights_:
                            v += tf.reduce_sum(weight_ * weight_) * self.p["dssm_l2"]
                
                if self.p["matrix_l2"]:
                    v += tf.reduce_sum(self.W * self.W) * self.p["matrix_l2"]
                
                if p["verbose"]:
                    print(v.numpy())
                
                return v

            weights = list()
            
            if self.train_matrix and (not self.p["Wfreeze"]):
                weights += [self.W]

            if self.train_dssm:
                weights += self.u_dssm.weights
                weights += self.g_dssm.weights
            
            if self.train_popbias:
                weights += [self.pb]

            if self.train_vbias:
                weights += [self.vb]   
                
            if self.train_bias:
                weights += [self.b]   
                
            
            if self.p["trainable_items"]:
                weights += [self.trainable_games]
                
                
            opt.minimize(loss, weights)
            
            if p["score_verbose"] and (i % p["score_verbose"] == 0):
                print(f"\n=== Iteration {i} ===")
                for sl in self.ctx.slices:
                    print(f"slice = {sl}, score = {self.get_score(sl)}")
        print("last loss = ", loss())

    def recommend(self, t):
        logits = 0
                    
        if self.train_matrix:
            logits += self.get_user_embs(t) @ self.W @ tf.transpose(self.get_game_embs())

        if self.train_dssm:
            logits += self.u_dssm(self.get_user_embs(t)) @ tf.transpose(self.g_dssm(self.get_game_embs()))

        if self.train_popbias:
            logits += self.pb * self.game_avg_scores["train"]       
            
        if self.train_vbias:
            logits += self.vb
            
        if self.train_bias:
            logits += self.b
            
        return logits
    
    # inherited
    # def get_score(self, t, topsize = 100):
    
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))
            
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))
            
class AnnCUR(Popular):
    def __init__(self, ctx, oracle=False, key_games=None, name=None):
        super().__init__(ctx)
        self.oracle = oracle
        self.ctx = ctx
        self.name = name

        self.key_cols_idx = np.array(sorted(ctx.key_games if key_games is None else key_games))
        rows_idx = np.arange(ctx.get_relevs("train").shape[0])
        
        self.cur = matrix_approx_zeshel.CURApprox(
            rows=torch.from_numpy(ctx.get_relevs("train")),
            cols=torch.from_numpy(ctx.get_relevs("train")[:, self.key_cols_idx]),
            row_idxs=rows_idx,
            col_idxs=self.key_cols_idx,
            approx_preference="rows",
            A=(torch.from_numpy(ctx.get_relevs("train")) if oracle else None)
        )        
        
    def __repr__(self):
        if self.name is None:
            return f"AnnCur({id(self)})"
        else:
            return f"AnnCur({self.name} - {id(self)})"
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        return ma.cur.latent_cols.T

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        key_relevs = torch.from_numpy(self.ctx.get_relevs(t)[:, self.key_cols_idx])
        return self.cur.get_complete_row(key_relevs)
    
    def get_score(self, t, topsize = 100):       
        recs = np.array(self.recommend(t))
        trus = self.get_user_scores(t)

        mse = np.mean((recs - trus) ** 2)

        recs = np.argsort(-recs, axis=1)[:, :topsize]
        trus = np.argsort(-trus, axis=1)[:, :topsize]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)

# True Games Data

In [5]:
L = 7000
N = 1000
DA = 50

ctx = load(L, raw_path = "//dev/null/stand/log.local.txt", path="log.local.bin.old", seed=17, det_attempts=DA)
# .set_top_games_as_key()
print("LOADED")

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.6280537283136858e-117
2.6280537283136858e-117
101 4644 2034
LOADED


In [6]:
ma = AnnCUR(ctx)

ma.fit()
ma.get_score("train"), ma.get_score("test")

np.mean(results), mse, len(results) =  0.6894724375538329 1.9673446972399546 4644
np.mean(results), mse, len(results) =  0.6770648967551622 2.2164459815281976 2034


(0.6894724375538329, 0.6770648967551622)

In [22]:
from sklearn.linear_model import Ridge

In [5]:

class Popular:
    def __init__(self, ctx):
        self.ctx = ctx
        self.game_avg_scores = {
            t : self.get_user_scores(t).mean(axis = 0)
            for t in self.ctx.slices
        }
        self.trus_top = dict()
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        return [self.top_logits for _ in self.get_user_scores(t)]
    
    def get_score(self, t, topsize = 100):    
        recs = self.recommend(t)
        
        if isinstance(recs, list):
            recs = np.array(recs)
        
        mse = np.mean((recs - self.get_user_scores(t)) ** 2)
        
        if isinstance(recs, tf.Tensor):
            recs = tf.argsort(-recs, axis=1)[:, :topsize].numpy()
        else:
            recs = np.argsort(-recs, axis=1)[:, :topsize]
        
        if (t, topsize) not in self.trus_top:
            trus = self.get_user_scores(t)
            trus = np.argsort(-trus, axis=1)[:, :topsize]
            self.trus_top[(t, topsize)] = trus
            
        trus = self.trus_top[(t, topsize)]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)
    

In [71]:
ctx.get_relevs("test").shape

(2034, 16514)

In [73]:
GE = ma.get_game_embs()

results = list()
K = 10
T = 100

for i in tqdm.tqdm(range(200)):
    A = ctx.key_games[:K]
    Rt = ctx.get_relevs("test")[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        R = Rt[A]
        Ae = GE[A]
        R, Ae.shape

        c = Ridge(fit_intercept=False)
        c.fit(Ae, R)

        Rp = c.predict(GE)
        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
        
    assert len(A) == T

    pred_top, gt_top = Rp.argsort()[::-1][:100], ctx.get_relevs("train")[i].argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

np.mean(results)

100%|██████████| 200/200 [01:02<00:00,  3.22it/s]


0.24109999999999998

In [75]:
GE = ctx.docblocks[6]

results = list()
K = 5
T = 100

for i in tqdm.tqdm(range(200)):
    A = ctx.key_games[:K]
    Rt = ctx.get_relevs("test")[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        R = Rt[A]
        Ae = GE[A]
        R, Ae.shape

        c = Ridge(fit_intercept=False)
        c.fit(Ae, R)

        Rp = c.predict(GE)
        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
        
    assert len(A) == T

    pred_top, gt_top = Rp.argsort()[::-1][:100], ctx.get_relevs("train")[i].argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

np.mean(results)

100%|██████████| 200/200 [01:14<00:00,  2.70it/s]


0.1493

In [77]:
GE = ma.get_game_embs()

results = list()
K = 1
T = 100

for i in tqdm.tqdm(range(200)):
    A = ctx.key_games[:K]
    Rt = ctx.get_relevs("test")[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        R = Rt[A]
        Ae = GE[A]
        R, Ae.shape

        c = Ridge(fit_intercept=False)
        c.fit(Ae, R)

        Rp = c.predict(GE)
        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
        
    assert len(A) == T

    pred_top, gt_top = Rp.argsort()[::-1][:100], ctx.get_relevs("train")[i].argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

np.mean(results)

100%|██████████| 200/200 [10:35<00:00,  3.18s/it]


0.2706

In [83]:
GE = ma.get_game_embs()

results = list()
K = 10
T = 100
R_All = ctx.get_relevs("test")

for i in tqdm.tqdm(range(R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        R = Rt[A]
        Ae = GE[A]
        R, Ae.shape

        c = Ridge(fit_intercept=False)
        c.fit(Ae, R)

        Rp = c.predict(GE)
        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
        
    assert len(A) == T

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

np.mean(results)

100%|██████████| 2034/2034 [09:45<00:00,  3.47it/s]


0.43618977384464114

In [90]:
GE = ma.get_game_embs().numpy()

results = list()
K = 10
T = 100
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(fit_intercept=False)
    c.fit(Ae, R)

    Rp = c.predict(GE)
    return Rp
    
for i in tqdm.tqdm(range(100 + 0 * R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        Rp = get_Rp(GE, A, Rt[A])
        
        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
        
    assert len(A) == T
    Rp = get_Rp(GE, A, Rt[A])

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

np.mean(results)

100%|██████████| 100/100 [00:16<00:00,  6.12it/s]


0.4607

In [89]:
type(ctx.docblocks[6])

numpy.ndarray

In [98]:
GE = ma.get_game_embs().numpy()

results = list()
K = 5
T = 100
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(fit_intercept=False)
    c.fit(Ae, R)

    Rp = c.predict(GE)
    return Rp
   
As = list()
for i in tqdm.tqdm(range(100 + 0 * R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        Rp = get_Rp(GE, A, Rt[A])
        
        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
        
    assert len(A) == T
    Rp = get_Rp(GE, A, Rt[A])
    As.append(A)

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

np.mean(results)

100%|██████████| 100/100 [00:27<00:00,  3.63it/s]


0.5021

In [99]:
GE = ctx.docblocks[6]

results = list()
K = 5
T = 100
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(fit_intercept=False)
    c.fit(Ae, R)

    Rp = c.predict(GE)
    return Rp
   
As = list()
for i in tqdm.tqdm(range(100 + 0 * R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        Rp = get_Rp(GE, A, Rt[A])
        
        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
        
    assert len(A) == T
    Rp = get_Rp(GE, A, Rt[A])
    As.append(A)

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

np.mean(results)

100%|██████████| 100/100 [00:35<00:00,  2.79it/s]


0.3282

In [100]:
GE = ma.get_game_embs().numpy()

results = list()
K = 1
T = 100
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(fit_intercept=False)
    c.fit(Ae, R)

    Rp = c.predict(GE)
    return Rp
   
As = list()
for i in tqdm.tqdm(range(100 + 0 * R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        Rp = get_Rp(GE, A, Rt[A])
        
        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
        
    assert len(A) == T
    Rp = get_Rp(GE, A, Rt[A])
    As.append(A)

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

np.mean(results)

100%|██████████| 100/100 [02:24<00:00,  1.44s/it]


0.5345

In [105]:
len(set(list(np.array(As).flatten())))

619

In [106]:
GE = ma.get_game_embs().numpy()

results = list()
K = 1
T = 100
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(fit_intercept=True)
    c.fit(Ae, R)

    Rp = c.predict(GE)
    return Rp
   
As = list()
for i in tqdm.tqdm(range(100 + 0 * R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        Rp = get_Rp(GE, A, Rt[A])
        
        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
        
    assert len(A) == T
    Rp = get_Rp(GE, A, Rt[A])
    As.append(A)

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

np.mean(results)

100%|██████████| 100/100 [02:35<00:00,  1.56s/it]


0.5613

In [107]:
GE = ma.get_game_embs().numpy()

results = list()
K = 5
T = 100
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(fit_intercept=True)
    c.fit(Ae, R)

    Rp = c.predict(GE)
    return Rp
   
As = list()
for i in tqdm.tqdm(range(100 + 0 * R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        Rp = get_Rp(GE, A, Rt[A])
        
        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
        
    assert len(A) == T
    Rp = get_Rp(GE, A, Rt[A])
    As.append(A)

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

np.mean(results)

100%|██████████| 100/100 [00:37<00:00,  2.68it/s]


0.5409

In [108]:
GE = ma.get_game_embs().numpy()

results = list()
K = 10
T = 100
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(fit_intercept=True)
    c.fit(Ae, R)

    Rp = c.predict(GE)
    return Rp
   
As = list()
for i in tqdm.tqdm(range(100 + 0 * R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        Rp = get_Rp(GE, A, Rt[A])
        
        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
        
    assert len(A) == T
    Rp = get_Rp(GE, A, Rt[A])
    As.append(A)

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

np.mean(results)

100%|██████████| 100/100 [00:18<00:00,  5.45it/s]


0.4605

In [109]:
GE = ctx.docblocks[6]

results = list()
K = 1
T = 100
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(fit_intercept=True)
    c.fit(Ae, R)

    Rp = c.predict(GE)
    return Rp
   
As = list()
for i in tqdm.tqdm(range(100 + 0 * R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        Rp = get_Rp(GE, A, Rt[A])
        
        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
        
    assert len(A) == T
    Rp = get_Rp(GE, A, Rt[A])
    As.append(A)

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

np.mean(results)

100%|██████████| 100/100 [02:33<00:00,  1.53s/it]


0.36230000000000007

In [94]:
GE = ctx.docblocks[6]

results = list()
K = 10
T = 100
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(alpha=2, fit_intercept=False)
    c.fit(Ae, R)

    Rp = c.predict(GE)
    return Rp
    
for i in tqdm.tqdm(range(100 + 0 * R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        Rp = get_Rp(GE, A, Rt[A])
        
        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
        
    assert len(A) == T
    Rp = get_Rp(GE, A, Rt[A])

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

np.mean(results)

100%|██████████| 100/100 [00:17<00:00,  5.82it/s]


0.27130000000000004

In [110]:
GE = ma.get_game_embs().numpy()

results = list()
K = 1
T = 100
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(fit_intercept=True)
    c.fit(Ae, R)

    Rp = c.predict(GE)
    return Rp
   
As = list()
for i in tqdm.tqdm(range(R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        Rp = get_Rp(GE, A, Rt[A])
        
        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
        
    assert len(A) == T
    Rp = get_Rp(GE, A, Rt[A])
    As.append(A)

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

np.mean(results)

 10%|█         | 204/2034 [05:04<45:34,  1.49s/it] 


KeyboardInterrupt: 

In [111]:
np.mean(results)

0.5466176470588235

In [131]:
GE = ma.get_game_embs().numpy()

results = list()
K = 1
T = 100
L = 0.1
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(fit_intercept=True)
    c.fit(Ae, R)

    Rp = c.predict(GE)
    

    return Rp
   
As = list()
for i in tqdm.tqdm(range(R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        Rp = get_Rp(GE, A, Rt[A])
        
        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
    assert len(A) == T
    Rp = L * get_Rp(GE, A, Rt[A]) + (1. - L) * (GE @ Rt[ma.key_cols_idx])
        
    As.append(A)

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

np.mean(results)

  5%|▍         | 100/2034 [02:35<50:03,  1.55s/it] 


KeyboardInterrupt: 

In [151]:
q = ctx.get_requests("test")


GE = ctx.docblocks[6]
QE = np.stack([q_i[1][6] for q_i in q])

results = list()
K = 1
T = 100
L = 0.5
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(fit_intercept=True)
    c.fit(Ae, R)

    Rp = c.predict(GE)
    

    return Rp
   
As = list()
for i in tqdm.tqdm(range(R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    if L != 0:
        for _ in range(T // K - 1):
            Rp = get_Rp(GE, A, Rt[A])

            An = list(A)
            for ai in Rp.argsort()[::-1]:
                if ai not in An:
                    An.append(ai)
                if len(An) == len(A) + K:
                    break
            A = np.array(An)
        assert len(A) == T

    Rp = L * get_Rp(GE, A, Rt[A]) + (1. - L) * (GE @ QE[i])
        
    As.append(A)

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

np.mean(results)

  3%|▎         | 70/2034 [01:41<47:19,  1.45s/it] 


KeyboardInterrupt: 

In [152]:
np.mean(results)

0.3747142857142857

In [147]:
(GE @ QE[i]).shape

(16514,)

array([0.        , 0.11111111, 0.22222222, 0.33333333, 0.44444444,
       0.55555556, 0.66666667, 0.77777778, 0.88888889, 1.        ])

In [157]:
for L in np.linspace(0, 1, 10):
    GE = ma.get_game_embs().numpy()

    results = list()
    K = 1
    T = 100
    R_All = ctx.get_relevs("test")

    def get_Rp(GE, A, R):
        Ae = GE[A]

        c = Ridge(fit_intercept=True)
        c.fit(Ae, R)

        Rp = c.predict(GE)


        return Rp

    As = list()
    for i in tqdm.tqdm(range(10 + 0 * R_All.shape[0])):
        A = ctx.key_games[:K]
        Rt = R_All[i]
        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

        for _ in range(T // K - 1):
            Rp = get_Rp(GE, A, Rt[A])

            An = list(A)
            for ai in Rp.argsort()[::-1]:
                if ai not in An:
                    An.append(ai)
                if len(An) == len(A) + K:
                    break
            A = np.array(An)
        assert len(A) == T
        Rp = L * get_Rp(GE, A, Rt[A]) + (1. - L) * (GE @ Rt[ma.key_cols_idx])

        As.append(A)

        pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
        results.append(ev(pred_top, gt_top) / float(100))

    print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████| 10/10 [00:16<00:00,  1.65s/it]


L = 0.0, np.mean(results) = 0.677


100%|██████████| 10/10 [00:15<00:00,  1.53s/it]


L = 0.1111111111111111, np.mean(results) = 0.681


100%|██████████| 10/10 [00:14<00:00,  1.44s/it]


...

L = 0.7777777777777777, np.mean(results) = 0.585


100%|██████████| 10/10 [00:12<00:00,  1.25s/it]


L = 0.8888888888888888, np.mean(results) = 0.575


100%|██████████| 10/10 [00:14<00:00,  1.41s/it]

L = 1.0, np.mean(results) = 0.5690000000000001


In [158]:
for L in np.linspace(0, 1, 10):
    GE = ma.get_game_embs().numpy()

    results = list()
    K = 1
    T = 100
    R_All = ctx.get_relevs("key")

    def get_Rp(GE, A, R):
        Ae = GE[A]

        c = Ridge(fit_intercept=True)
        c.fit(Ae, R)

        Rp = c.predict(GE)


        return Rp

    As = list()
    for i in tqdm.tqdm(range(R_All.shape[0])):
        A = ctx.key_games[:K]
        Rt = R_All[i]
        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

        for _ in range(T // K - 1):
            Rp = get_Rp(GE, A, Rt[A])

            An = list(A)
            for ai in Rp.argsort()[::-1]:
                if ai not in An:
                    An.append(ai)
                if len(An) == len(A) + K:
                    break
            A = np.array(An)
        assert len(A) == T
        Rp = L * get_Rp(GE, A, Rt[A]) + (1. - L) * (GE @ Rt[ma.key_cols_idx])

        As.append(A)

        pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
        results.append(ev(pred_top, gt_top) / float(100))

    print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████| 101/101 [02:31<00:00,  1.50s/it]


L = 0.0, np.mean(results) = 0.6553465346534654


100%|██████████| 101/101 [02:24<00:00,  1.43s/it]


L = 0.1111111111111111, np.mean(results) = 0.6297029702970298


100%|██████████| 101/101 [02:16<00:00,  1.36s/it]


...

L = 0.7777777777777777, np.mean(results) = 0.5761386138613861


100%|██████████| 101/101 [02:24<00:00,  1.43s/it]


L = 0.8888888888888888, np.mean(results) = 0.5675247524752475


100%|██████████| 101/101 [02:31<00:00,  1.50s/it]

L = 1.0, np.mean(results) = 0.5594059405940595


In [159]:
for L in np.linspace(0, 1, 10):
    GE = ma.get_game_embs().numpy()

    results = list()
    K = 1
    T = 100
    R_All = ctx.get_relevs("key")

    def get_Rp(GE, A, R):
        Ae = GE[A]

        c = Ridge(fit_intercept=True)
        c.fit(Ae, R)

        Rp = c.predict(GE)


        return Rp

    As = list()
    for i in tqdm.tqdm(range(R_All.shape[0])):
        A = ctx.key_games[:K]
        Rt = R_All[i]
        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

        for _ in range(T // K - 1):
            Rp = get_Rp(GE, A, Rt[A])

            An = list(A)
            for ai in Rp.argsort()[::-1]:
                if ai not in An:
                    An.append(ai)
                if len(An) == len(A) + K:
                    break
            A = np.array(An)
        assert len(A) == T
        
        Rp1 = get_Rp(GE, A, Rt[A])
        Rp2 = GE @ Rt[ma.key_cols_idx]
        
        Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
        Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()
        
        Rp = L * Rp1 + (1. - L) * Rp2

        As.append(A)

        pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
        results.append(ev(pred_top, gt_top) / float(100))

    print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████| 101/101 [02:09<00:00,  1.28s/it]


L = 0.0, np.mean(results) = 0.6553465346534654


100%|██████████| 101/101 [02:10<00:00,  1.29s/it]


L = 0.1111111111111111, np.mean(results) = 0.6675247524752476


100%|██████████| 101/101 [02:15<00:00,  1.35s/it]


...

L = 0.5555555555555556, np.mean(results) = 0.65009900990099


100%|██████████| 101/101 [02:26<00:00,  1.45s/it]


L = 0.6666666666666666, np.mean(results) = 0.6323762376237623


 30%|██▉       | 30/101 [00:41<01:37,  1.38s/it]


KeyboardInterrupt: 

In [160]:
for L in np.linspace(0, 1, 10):
    GE = ma.get_game_embs().numpy()

    results = list()
    K = 5
    T = 100
    R_All = ctx.get_relevs("key")

    def get_Rp(GE, A, R):
        Ae = GE[A]

        c = Ridge(fit_intercept=True)
        c.fit(Ae, R)

        Rp = c.predict(GE)


        return Rp

    As = list()
    for i in tqdm.tqdm(range(R_All.shape[0])):
        A = ctx.key_games[:K]
        Rt = R_All[i]
        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

        for _ in range(T // K - 1):
            Rp = get_Rp(GE, A, Rt[A])

            An = list(A)
            for ai in Rp.argsort()[::-1]:
                if ai not in An:
                    An.append(ai)
                if len(An) == len(A) + K:
                    break
            A = np.array(An)
        assert len(A) == T
        
        Rp1 = get_Rp(GE, A, Rt[A])
        Rp2 = GE @ Rt[ma.key_cols_idx]
        
        Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
        Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()
        
        Rp = L * Rp1 + (1. - L) * Rp2

        As.append(A)

        pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
        results.append(ev(pred_top, gt_top) / float(100))

    print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████| 101/101 [00:34<00:00,  2.94it/s]


L = 0.0, np.mean(results) = 0.6553465346534654


100%|██████████| 101/101 [00:38<00:00,  2.62it/s]


L = 0.1111111111111111, np.mean(results) = 0.6696039603960395


100%|██████████| 101/101 [00:33<00:00,  3.06it/s]


L = 0.2222222222222222, np.mean(results) = 0.6743564356435644


100%|██████████| 101/101 [00:33<00:00,  3.04it/s]


L = 0.3333333333333333, np.mean(results) = 0.6715841584158417


  6%|▌         | 6/101 [00:02<00:33,  2.82it/s]


KeyboardInterrupt: 

In [164]:
L = 2./9
GE = ma.get_game_embs().numpy()

results = list()
K = 5
T = 100
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(fit_intercept=True)
    c.fit(Ae, R)

    Rp = c.predict(GE)


    return Rp

As = list()
for i in tqdm.tqdm(range(R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        Rp = get_Rp(GE, A, Rt[A])

        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
    assert len(A) == T

    Rp1 = get_Rp(GE, A, Rt[A])
    Rp2 = GE @ Rt[ma.key_cols_idx]

    Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
    Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()

    Rp = L * Rp1 + (1. - L) * Rp2

    As.append(A)

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████| 2034/2034 [11:40<00:00,  2.90it/s]

L = 0.2222222222222222, np.mean(results) = 0.6934070796460177


In [165]:
ctx.set_kmeans_games_as_key()

ma = AnnCUR(ctx)

ma.fit()
ma.get_score("train"), ma.get_score("test")

np.mean(results), mse, len(results) =  0.7200904392764859 0.18808463675768178 4644
np.mean(results), mse, len(results) =  0.7132005899705015 0.2785226137457979 2034


(0.7200904392764859, 0.7132005899705015)

In [167]:
ma.get_score("key")

np.mean(results), mse, len(results) =  0.6850495049504952 0.41383742781954436 101


0.6850495049504952

In [168]:
for L in np.linspace(0, 1, 10):
    GE = ma.get_game_embs().numpy()

    results = list()
    K = 5
    T = 100
    R_All = ctx.get_relevs("key")

    def get_Rp(GE, A, R):
        Ae = GE[A]

        c = Ridge(fit_intercept=True)
        c.fit(Ae, R)

        Rp = c.predict(GE)


        return Rp

    As = list()
    for i in tqdm.tqdm(range(R_All.shape[0])):
        A = ctx.key_games[:K]
        Rt = R_All[i]
        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

        for _ in range(T // K - 1):
            Rp = get_Rp(GE, A, Rt[A])

            An = list(A)
            for ai in Rp.argsort()[::-1]:
                if ai not in An:
                    An.append(ai)
                if len(An) == len(A) + K:
                    break
            A = np.array(An)
        assert len(A) == T
        
        Rp1 = get_Rp(GE, A, Rt[A])
        Rp2 = GE @ Rt[ma.key_cols_idx]
        
        Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
        Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()
        
        Rp = L * Rp1 + (1. - L) * Rp2

        As.append(A)

        pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
        results.append(ev(pred_top, gt_top) / float(100))

    print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████| 101/101 [00:37<00:00,  2.69it/s]


L = 0.0, np.mean(results) = 0.6850495049504952


100%|██████████| 101/101 [00:36<00:00,  2.80it/s]


L = 0.1111111111111111, np.mean(results) = 0.6831683168316832


100%|██████████| 101/101 [00:34<00:00,  2.91it/s]


...

L = 0.3333333333333333, np.mean(results) = 0.6646534653465348


100%|██████████| 101/101 [00:33<00:00,  3.02it/s]


L = 0.4444444444444444, np.mean(results) = 0.6496039603960394


 31%|███       | 31/101 [00:08<00:18,  3.87it/s]


KeyboardInterrupt: 

In [169]:
L = 2./9
GE = ma.get_game_embs().numpy()

results = list()
K = 5
T = 100
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(fit_intercept=True)
    c.fit(Ae, R)

    Rp = c.predict(GE)


    return Rp

As = list()
for i in tqdm.tqdm(range(R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        Rp = get_Rp(GE, A, Rt[A])

        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
    assert len(A) == T

    Rp1 = get_Rp(GE, A, Rt[A])
    Rp2 = GE @ Rt[ma.key_cols_idx]

    Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
    Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()

    Rp = L * Rp1 + (1. - L) * Rp2

    As.append(A)

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████| 2034/2034 [11:06<00:00,  3.05it/s]

L = 0.2222222222222222, np.mean(results) = 0.702409046214356


In [12]:
def coitem_algorithm(n_support, candidate_items, target_items, min_item_rel_norm=1e-5, eps=1e-6):
    support_items = []
    support_items_scores = []
    n_c, n_q = candidate_items.shape
    n_t = target_items.shape[0]

    candidate_item_squared_norms = (candidate_items ** 2).sum(axis=1)
    min_item_norm = min_item_rel_norm * np.sqrt(candidate_item_squared_norms.mean())
    
    candidate_coitems = candidate_items.dot(
        target_items.T.dot(target_items)
    )
    orthonormed_support_items = np.zeros((n_support, n_q))
    remaining_items = np.ones(candidate_items.shape[0], dtype="bool")
    for t in tqdm.tqdm_notebook(range(n_support)):
        scores = (candidate_coitems * candidate_items).sum(axis=1)
        remaining_items &= (candidate_item_squared_norms >= min_item_norm ** 2)
        scores[remaining_items] /= candidate_item_squared_norms[remaining_items]
        scores[~remaining_items] = 0
        
        new_item_id = np.argmax(scores)
        assert remaining_items[new_item_id]
        support_items.append(new_item_id)
        support_items_scores.append(scores[new_item_id] / (n_t * n_q))
        remaining_items[new_item_id] = False
        
        new_item = candidate_items[new_item_id].copy()
        new_item -= orthonormed_support_items[:t].T.dot(
            orthonormed_support_items[:t].dot(new_item)
        )
        assert np.allclose((new_item ** 2).sum(), candidate_item_squared_norms[new_item_id], atol=eps)
        new_item /= np.sqrt((new_item ** 2).sum())
        orthonormed_support_items[t] = new_item
        new_coitem = candidate_coitems[new_item_id] / np.sqrt(candidate_item_squared_norms[new_item_id])
        
        coefs = (candidate_items * new_item).sum(axis=1)
        candidate_item_squared_norms -= coefs ** 2
        assert np.all(candidate_item_squared_norms[remaining_items] > -eps)
        
        candidate_coitems -= coefs.reshape((-1, 1)).dot(new_coitem.reshape((1, -1)))
        cocoefs = (candidate_coitems * new_item).sum(axis=1, keepdims=True)
        candidate_coitems -= cocoefs.dot(new_item.reshape((1, -1)))
    return np.array(support_items, dtype=np.int64), np.array(support_items_scores)

In [172]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("key"), ma.get_score("train"), ma.get_score("test")

<ipython-input-171-a13f9b2faac9>:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.7066336633663365 0.36810762681519194 101
np.mean(results), mse, len(results) =  0.7339233419465978 0.17275461043851922 4644
np.mean(results), mse, len(results) =  0.7261406096361849 0.2692821086659797 2034


(0.7066336633663365, 0.7339233419465978, 0.7261406096361849)

In [173]:
for L in np.linspace(0, 1, 10):
    GE = ma.get_game_embs().numpy()

    results = list()
    K = 5
    T = 100
    R_All = ctx.get_relevs("key")

    def get_Rp(GE, A, R):
        Ae = GE[A]

        c = Ridge(fit_intercept=True)
        c.fit(Ae, R)

        Rp = c.predict(GE)


        return Rp

    As = list()
    for i in tqdm.tqdm(range(R_All.shape[0])):
        A = ctx.key_games[:K]
        Rt = R_All[i]
        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

        for _ in range(T // K - 1):
            Rp = get_Rp(GE, A, Rt[A])

            An = list(A)
            for ai in Rp.argsort()[::-1]:
                if ai not in An:
                    An.append(ai)
                if len(An) == len(A) + K:
                    break
            A = np.array(An)
        assert len(A) == T
        
        Rp1 = get_Rp(GE, A, Rt[A])
        Rp2 = GE @ Rt[ma.key_cols_idx]
        
        Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
        Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()
        
        Rp = L * Rp1 + (1. - L) * Rp2

        As.append(A)

        pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
        results.append(ev(pred_top, gt_top) / float(100))

    print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████| 101/101 [00:37<00:00,  2.70it/s]


L = 0.0, np.mean(results) = 0.7066336633663365


100%|██████████| 101/101 [00:38<00:00,  2.63it/s]


L = 0.1111111111111111, np.mean(results) = 0.6996039603960394


100%|██████████| 101/101 [00:34<00:00,  2.91it/s]


...

L = 0.7777777777777777, np.mean(results) = 0.5957425742574257


100%|██████████| 101/101 [00:36<00:00,  2.76it/s]


L = 0.8888888888888888, np.mean(results) = 0.5721782178217822


100%|██████████| 101/101 [00:31<00:00,  3.24it/s]

L = 1.0, np.mean(results) = 0.539009900990099


In [174]:
L = 2./9
GE = ma.get_game_embs().numpy()

results = list()
K = 5
T = 100
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(fit_intercept=True)
    c.fit(Ae, R)

    Rp = c.predict(GE)


    return Rp

As = list()
for i in tqdm.tqdm(range(R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        Rp = get_Rp(GE, A, Rt[A])

        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
    assert len(A) == T

    Rp1 = get_Rp(GE, A, Rt[A])
    Rp2 = GE @ Rt[ma.key_cols_idx]

    Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
    Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()

    Rp = L * Rp1 + (1. - L) * Rp2

    As.append(A)

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████| 2034/2034 [12:00<00:00,  2.82it/s]

L = 0.2222222222222222, np.mean(results) = 0.7080137659783677


In [ ]:
L = 2./9
GE = ma.get_game_embs().numpy()

results = list()
K = 5
T = 100
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(fit_intercept=True)
    c.fit(Ae, R)

    Rp = c.predict(GE)


    return Rp

As = list()
for i in tqdm.tqdm(range(R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        Rp = get_Rp(GE, A, Rt[A])

        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
    assert len(A) == T

    Rp1 = get_Rp(GE, A, Rt[A])
    Rp2 = GE @ Rt[ma.key_cols_idx]

    Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
    Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()

    Rp = L * Rp1 + (1. - L) * Rp2

    As.append(A)

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

print(f"L = {L}, np.mean(results) = {np.mean(results)}")

### other games data

In [177]:
L = 7000
ctx = load(L, raw_path = "stand/log.local.logtime2.txt",
           path="log.local.logtime2.true.bin", seed=17, det_attempts=DA)

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.6095219944863732e-120
2.6095219944863732e-120
101 4769 2088


In [178]:
ma = AnnCUR(ctx)

ma.fit()
ma.get_score("key"), ma.get_score("train"), ma.get_score("test")

np.mean(results), mse, len(results) =  0.5700990099009902 0.2015431043540931 101
np.mean(results), mse, len(results) =  0.5958691549591109 0.17399199764984383 4769
np.mean(results), mse, len(results) =  0.5852155172413792 0.19193542843992634 2088


(0.5700990099009902, 0.5958691549591109, 0.5852155172413792)

In [179]:
for L in np.linspace(0, 1, 10):
    GE = ma.get_game_embs().numpy()

    results = list()
    K = 5
    T = 100
    R_All = ctx.get_relevs("key")

    def get_Rp(GE, A, R):
        Ae = GE[A]

        c = Ridge(fit_intercept=True)
        c.fit(Ae, R)

        Rp = c.predict(GE)


        return Rp

    As = list()
    for i in tqdm.tqdm(range(R_All.shape[0])):
        A = ctx.key_games[:K]
        Rt = R_All[i]
        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

        for _ in range(T // K - 1):
            Rp = get_Rp(GE, A, Rt[A])

            An = list(A)
            for ai in Rp.argsort()[::-1]:
                if ai not in An:
                    An.append(ai)
                if len(An) == len(A) + K:
                    break
            A = np.array(An)
        assert len(A) == T
        
        Rp1 = get_Rp(GE, A, Rt[A])
        Rp2 = GE @ Rt[ma.key_cols_idx]
        
        Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
        Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()
        
        Rp = L * Rp1 + (1. - L) * Rp2

        As.append(A)

        pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
        results.append(ev(pred_top, gt_top) / float(100))

    print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████| 101/101 [00:36<00:00,  2.75it/s]


L = 0.0, np.mean(results) = 0.5700990099009902


100%|██████████| 101/101 [00:31<00:00,  3.20it/s]


L = 0.1111111111111111, np.mean(results) = 0.5984158415841584


100%|██████████| 101/101 [00:35<00:00,  2.86it/s]


...

L = 0.7777777777777777, np.mean(results) = 0.5097029702970298


100%|██████████| 101/101 [00:36<00:00,  2.76it/s]


L = 0.8888888888888888, np.mean(results) = 0.49544554455445555


100%|██████████| 101/101 [00:36<00:00,  2.73it/s]

L = 1.0, np.mean(results) = 0.4808910891089109


In [180]:
L = 2./9
GE = ma.get_game_embs().numpy()

results = list()
K = 5
T = 100
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(fit_intercept=True)
    c.fit(Ae, R)

    Rp = c.predict(GE)


    return Rp

As = list()
for i in tqdm.tqdm(range(R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        Rp = get_Rp(GE, A, Rt[A])

        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
    assert len(A) == T

    Rp1 = get_Rp(GE, A, Rt[A])
    Rp2 = GE @ Rt[ma.key_cols_idx]

    Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
    Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()

    Rp = L * Rp1 + (1. - L) * Rp2

    As.append(A)

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████| 2088/2088 [12:08<00:00,  2.86it/s]

L = 0.2222222222222222, np.mean(results) = 0.6153304597701149


In [181]:
for L in np.linspace(0, 1, 11):
    GE = ma.get_game_embs().numpy()

    results = list()
    K = 5
    T = 100
    R_All = ctx.get_relevs("key")

    def get_Rp(GE, A, R):
        Ae = GE[A]

        c = Ridge(fit_intercept=True)
        c.fit(Ae, R)

        Rp = c.predict(GE)


        return Rp

    As = list()
    for i in tqdm.tqdm(range(R_All.shape[0])):
        A = ctx.key_games[:K]
        Rt = R_All[i]
        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

        for _ in range(T // K - 1):
            Rp = get_Rp(GE, A, Rt[A])

            An = list(A)
            for ai in Rp.argsort()[::-1]:
                if ai not in An:
                    An.append(ai)
                if len(An) == len(A) + K:
                    break
            A = np.array(An)
        assert len(A) == T
        
        Rp1 = get_Rp(GE, A, Rt[A])
        Rp2 = GE @ Rt[ma.key_cols_idx]
        
        Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
        Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()
        
        Rp = L * Rp1 + (1. - L) * Rp2

        As.append(A)

        pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
        results.append(ev(pred_top, gt_top) / float(100))

    print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████| 101/101 [00:36<00:00,  2.75it/s]


L = 0.0, np.mean(results) = 0.5700990099009902


100%|██████████| 101/101 [00:35<00:00,  2.85it/s]


L = 0.1, np.mean(results) = 0.5971287128712871


100%|██████████| 101/101 [00:34<00:00,  2.96it/s]


...

L = 0.8, np.mean(results) = 0.5066336633663365


100%|██████████| 101/101 [00:32<00:00,  3.12it/s]


L = 0.9, np.mean(results) = 0.49346534653465357


100%|██████████| 101/101 [00:36<00:00,  2.79it/s]

L = 1.0, np.mean(results) = 0.4808910891089109


In [182]:
L = 0.2
GE = ma.get_game_embs().numpy()

results = list()
K = 5
T = 100
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(fit_intercept=True)
    c.fit(Ae, R)

    Rp = c.predict(GE)


    return Rp

As = list()
for i in tqdm.tqdm(range(R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        Rp = get_Rp(GE, A, Rt[A])

        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
    assert len(A) == T

    Rp1 = get_Rp(GE, A, Rt[A])
    Rp2 = GE @ Rt[ma.key_cols_idx]

    Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
    Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()

    Rp = L * Rp1 + (1. - L) * Rp2

    As.append(A)

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████| 2088/2088 [12:20<00:00,  2.82it/s]

L = 0.2, np.mean(results) = 0.6170545977011495


In [184]:
ctx.set_kmeans_games_as_key()

ma = AnnCUR(ctx)

ma.fit()
ma.get_score("key"), ma.get_score("train"), ma.get_score("test")

np.mean(results), mse, len(results) =  0.6074257425742574 0.18763741888016733 101
np.mean(results), mse, len(results) =  0.6287607464877334 0.15658457174869786 4769
np.mean(results), mse, len(results) =  0.6183620689655172 0.173750329075636 2088


(0.6074257425742574, 0.6287607464877334, 0.6183620689655172)

In [185]:
for L in np.linspace(0, 1, 11):
    GE = ma.get_game_embs().numpy()

    results = list()
    K = 5
    T = 100
    R_All = ctx.get_relevs("key")

    def get_Rp(GE, A, R):
        Ae = GE[A]

        c = Ridge(fit_intercept=True)
        c.fit(Ae, R)

        Rp = c.predict(GE)


        return Rp

    As = list()
    for i in tqdm.tqdm(range(R_All.shape[0])):
        A = ctx.key_games[:K]
        Rt = R_All[i]
        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

        for _ in range(T // K - 1):
            Rp = get_Rp(GE, A, Rt[A])

            An = list(A)
            for ai in Rp.argsort()[::-1]:
                if ai not in An:
                    An.append(ai)
                if len(An) == len(A) + K:
                    break
            A = np.array(An)
        assert len(A) == T
        
        Rp1 = get_Rp(GE, A, Rt[A])
        Rp2 = GE @ Rt[ma.key_cols_idx]
        
        Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
        Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()
        
        Rp = L * Rp1 + (1. - L) * Rp2

        As.append(A)

        pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
        results.append(ev(pred_top, gt_top) / float(100))

    print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████| 101/101 [00:37<00:00,  2.71it/s]


L = 0.0, np.mean(results) = 0.6074257425742574


100%|██████████| 101/101 [00:33<00:00,  3.06it/s]


L = 0.1, np.mean(results) = 0.6297029702970296


100%|██████████| 101/101 [00:35<00:00,  2.86it/s]


...

L = 0.8, np.mean(results) = 0.47108910891089106


100%|██████████| 101/101 [00:35<00:00,  2.85it/s]


L = 0.9, np.mean(results) = 0.4537623762376237


100%|██████████| 101/101 [00:34<00:00,  2.91it/s]

L = 1.0, np.mean(results) = 0.43891089108910897


In [187]:
L = 0.1
GE = ma.get_game_embs().numpy()

results = list()
K = 5
T = 100
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(fit_intercept=True)
    c.fit(Ae, R)

    Rp = c.predict(GE)


    return Rp

As = list()
for i in tqdm.tqdm(range(R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        Rp = get_Rp(GE, A, Rt[A])

        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
    assert len(A) == T

    Rp1 = get_Rp(GE, A, Rt[A])
    Rp2 = GE @ Rt[ma.key_cols_idx]

    Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
    Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()

    Rp = L * Rp1 + (1. - L) * Rp2

    As.append(A)

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████| 2088/2088 [12:25<00:00,  2.80it/s]

L = 0.1, np.mean(results) = 0.6388936781609197


In [188]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("key"), ma.get_score("train"), ma.get_score("test")

<ipython-input-171-a13f9b2faac9>:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.6531683168316833 0.16510736271699725 101
np.mean(results), mse, len(results) =  0.6684629901446845 0.13094301466182667 4769
np.mean(results), mse, len(results) =  0.6609386973180077 0.14711578383534862 2088


(0.6531683168316833, 0.6684629901446845, 0.6609386973180077)

In [189]:
for L in np.linspace(0, 1, 11):
    GE = ma.get_game_embs().numpy()

    results = list()
    K = 5
    T = 100
    R_All = ctx.get_relevs("key")

    def get_Rp(GE, A, R):
        Ae = GE[A]

        c = Ridge(fit_intercept=True)
        c.fit(Ae, R)

        Rp = c.predict(GE)


        return Rp

    As = list()
    for i in tqdm.tqdm(range(R_All.shape[0])):
        A = ctx.key_games[:K]
        Rt = R_All[i]
        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

        for _ in range(T // K - 1):
            Rp = get_Rp(GE, A, Rt[A])

            An = list(A)
            for ai in Rp.argsort()[::-1]:
                if ai not in An:
                    An.append(ai)
                if len(An) == len(A) + K:
                    break
            A = np.array(An)
        assert len(A) == T
        
        Rp1 = get_Rp(GE, A, Rt[A])
        Rp2 = GE @ Rt[ma.key_cols_idx]
        
        Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
        Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()
        
        Rp = L * Rp1 + (1. - L) * Rp2

        As.append(A)

        pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
        results.append(ev(pred_top, gt_top) / float(100))

    print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████| 101/101 [00:32<00:00,  3.09it/s]


L = 0.0, np.mean(results) = 0.6531683168316833


100%|██████████| 101/101 [00:26<00:00,  3.79it/s]


L = 0.1, np.mean(results) = 0.6643564356435644


100%|██████████| 101/101 [00:32<00:00,  3.10it/s]


...

L = 0.8, np.mean(results) = 0.4896039603960397


100%|██████████| 101/101 [00:36<00:00,  2.75it/s]


L = 0.9, np.mean(results) = 0.47158415841584156


100%|██████████| 101/101 [00:34<00:00,  2.92it/s]

L = 1.0, np.mean(results) = 0.4543564356435644


In [191]:
L = 0.1
GE = ma.get_game_embs().numpy()

results = list()
K = 5
T = 100
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(fit_intercept=True)
    c.fit(Ae, R)

    Rp = c.predict(GE)


    return Rp

As = list()
for i in tqdm.tqdm(range(R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        Rp = get_Rp(GE, A, Rt[A])

        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
    assert len(A) == T

    Rp1 = get_Rp(GE, A, Rt[A])
    Rp2 = GE @ Rt[ma.key_cols_idx]

    Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
    Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()

    Rp = L * Rp1 + (1. - L) * Rp2

    As.append(A)

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████| 2088/2088 [11:32<00:00,  3.01it/s]

L = 0.1, np.mean(results) = 0.6734722222222222


In [192]:
t = ctx.get_relevs("train").T
t = (t - t.mean()) / t.std()  # like in other comparisons with RBE
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("key"), ma.get_score("train"), ma.get_score("test")

<ipython-input-171-a13f9b2faac9>:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.6510891089108911 0.15728289408259197 101
np.mean(results), mse, len(results) =  0.6655063954707486 0.1316394167164136 4769
np.mean(results), mse, len(results) =  0.6565373563218391 0.147431526765195 2088


(0.6510891089108911, 0.6655063954707486, 0.6565373563218391)

In [193]:
for L in np.linspace(0, 1, 11):
    GE = ma.get_game_embs().numpy()

    results = list()
    K = 5
    T = 100
    R_All = ctx.get_relevs("key")

    def get_Rp(GE, A, R):
        Ae = GE[A]

        c = Ridge(fit_intercept=True)
        c.fit(Ae, R)

        Rp = c.predict(GE)


        return Rp

    As = list()
    for i in tqdm.tqdm(range(R_All.shape[0])):
        A = ctx.key_games[:K]
        Rt = R_All[i]
        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

        for _ in range(T // K - 1):
            Rp = get_Rp(GE, A, Rt[A])

            An = list(A)
            for ai in Rp.argsort()[::-1]:
                if ai not in An:
                    An.append(ai)
                if len(An) == len(A) + K:
                    break
            A = np.array(An)
        assert len(A) == T
        
        Rp1 = get_Rp(GE, A, Rt[A])
        Rp2 = GE @ Rt[ma.key_cols_idx]
        
        Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
        Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()
        
        Rp = L * Rp1 + (1. - L) * Rp2

        As.append(A)

        pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
        results.append(ev(pred_top, gt_top) / float(100))

    print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████| 101/101 [00:39<00:00,  2.53it/s]


L = 0.0, np.mean(results) = 0.6510891089108911


100%|██████████| 101/101 [00:34<00:00,  2.91it/s]


L = 0.1, np.mean(results) = 0.6625742574257425


100%|██████████| 101/101 [00:36<00:00,  2.74it/s]


...

L = 0.8, np.mean(results) = 0.46673267326732676


100%|██████████| 101/101 [00:38<00:00,  2.62it/s]


L = 0.9, np.mean(results) = 0.4473267326732673


100%|██████████| 101/101 [00:35<00:00,  2.86it/s]

L = 1.0, np.mean(results) = 0.4302970297029704


In [194]:
L = 0.1
GE = ma.get_game_embs().numpy()

results = list()
K = 5
T = 100
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(fit_intercept=True)
    c.fit(Ae, R)

    Rp = c.predict(GE)


    return Rp

As = list()
for i in tqdm.tqdm(range(R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        Rp = get_Rp(GE, A, Rt[A])

        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
    assert len(A) == T

    Rp1 = get_Rp(GE, A, Rt[A])
    Rp2 = GE @ Rt[ma.key_cols_idx]

    Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
    Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()

    Rp = L * Rp1 + (1. - L) * Rp2

    As.append(A)

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████| 2088/2088 [11:31<00:00,  3.02it/s]

L = 0.1, np.mean(results) = 0.668146551724138


## games data back

In [6]:
L = 7000
N = 1000
DA = 50

ctx = load(L, raw_path = "//dev/null/stand/log.local.txt", path="log.local.bin.old", seed=17, det_attempts=DA)
# .set_top_games_as_key()
print("LOADED")

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.628053728316076e-117
2.628053728316076e-117
101 4644 2034
LOADED


In [6]:
ma = AnnCUR(ctx)

ma.fit()
ma.get_score("key"), ma.get_score("train"), ma.get_score("test")

np.mean(results), mse, len(results) =  0.6586138613861388 4.137429064391828 101
np.mean(results), mse, len(results) =  0.6835335917312662 1.9419027705784624 4644
np.mean(results), mse, len(results) =  0.6720894788593904 2.1916676320782016 2034


(0.6586138613861388, 0.6835335917312662, 0.6720894788593904)

In [8]:
from sklearn.linear_model import Ridge

for L in np.linspace(0, 1, 11):
    GE = ma.get_game_embs().numpy()

    results = list()
    K = 5
    T = 100
    R_All = ctx.get_relevs("key")

    def get_Rp(GE, A, R):
        Ae = GE[A]

        c = Ridge(fit_intercept=True)
        c.fit(Ae, R)

        Rp = c.predict(GE)


        return Rp

    As = list()
    for i in tqdm.tqdm(range(R_All.shape[0])):
        A = ctx.key_games[:K]
        Rt = R_All[i]
        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

        for _ in range(T // K - 1):
            Rp = get_Rp(GE, A, Rt[A])

            An = list(A)
            for ai in Rp.argsort()[::-1]:
                if ai not in An:
                    An.append(ai)
                if len(An) == len(A) + K:
                    break
            A = np.array(An)
        assert len(A) == T
        
        Rp1 = get_Rp(GE, A, Rt[A])
        Rp2 = GE @ Rt[ma.key_cols_idx]
        
        Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
        Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()
        
        Rp = L * Rp1 + (1. - L) * Rp2

        As.append(A)

        pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
        results.append(ev(pred_top, gt_top) / float(100))

    print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████| 101/101 [00:35<00:00,  2.81it/s]


L = 0.0, np.mean(results) = 0.6586138613861388


100%|██████████| 101/101 [00:36<00:00,  2.74it/s]


L = 0.1, np.mean(results) = 0.6698019801980198


100%|██████████| 101/101 [00:33<00:00,  2.99it/s]


...

L = 0.8, np.mean(results) = 0.5915841584158417


100%|██████████| 101/101 [00:35<00:00,  2.88it/s]


L = 0.9, np.mean(results) = 0.5687128712871287


100%|██████████| 101/101 [00:36<00:00,  2.74it/s]

L = 1.0, np.mean(results) = 0.5421782178217822


In [9]:
L = 0.2
GE = ma.get_game_embs().numpy()

results = list()
K = 5
T = 100
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(fit_intercept=True)
    c.fit(Ae, R)

    Rp = c.predict(GE)

    return Rp

As = list()
for i in tqdm.tqdm(range(R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        Rp = get_Rp(GE, A, Rt[A])

        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
    assert len(A) == T

    Rp1 = get_Rp(GE, A, Rt[A])
    Rp2 = GE @ Rt[ma.key_cols_idx]

    Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
    Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()

    Rp = L * Rp1 + (1. - L) * Rp2

    As.append(A)

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████| 2034/2034 [16:25<00:00,  2.06it/s]

L = 0.2, np.mean(results) = 0.6912094395280236


In [17]:
ctx.set_kmeans_games_as_key()

ma = AnnCUR(ctx)

ma.fit()
ma.get_score("key"), ma.get_score("train"), ma.get_score("test")

/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/base.py:1151: ConvergenceWarning: Number of distinct clusters (95) found smaller than n_clusters (100). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


np.mean(results), mse, len(results) =  0.7029702970297029 1.0476293520055127 101
np.mean(results), mse, len(results) =  0.7352820844099913 0.7430246420788484 4644
np.mean(results), mse, len(results) =  0.7292871189773845 1.056577488466122 2034


(0.7029702970297029, 0.7352820844099913, 0.7292871189773845)

In [11]:
from sklearn.linear_model import Ridge

for L in np.linspace(0, 1, 11):
    GE = ma.get_game_embs().numpy()

    results = list()
    K = 5
    T = 100
    R_All = ctx.get_relevs("key")

    def get_Rp(GE, A, R):
        Ae = GE[A]

        c = Ridge(fit_intercept=True)
        c.fit(Ae, R)

        Rp = c.predict(GE)


        return Rp

    As = list()
    for i in tqdm.tqdm(range(R_All.shape[0])):
        A = ctx.key_games[:K]
        Rt = R_All[i]
        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

        for _ in range(T // K - 1):
            Rp = get_Rp(GE, A, Rt[A])

            An = list(A)
            for ai in Rp.argsort()[::-1]:
                if ai not in An:
                    An.append(ai)
                if len(An) == len(A) + K:
                    break
            A = np.array(An)
        assert len(A) == T
        
        Rp1 = get_Rp(GE, A, Rt[A])
        Rp2 = GE @ Rt[ma.key_cols_idx]
        
        Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
        Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()
        
        Rp = L * Rp1 + (1. - L) * Rp2

        As.append(A)

        pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
        results.append(ev(pred_top, gt_top) / float(100))

    print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████| 101/101 [02:06<00:00,  1.25s/it]


L = 0.0, np.mean(results) = 0.6850495049504952


100%|██████████| 101/101 [02:08<00:00,  1.27s/it]


L = 0.1, np.mean(results) = 0.6838613861386138


100%|██████████| 101/101 [02:06<00:00,  1.25s/it]


...

L = 0.8, np.mean(results) = 0.5854455445544555


100%|██████████| 101/101 [01:52<00:00,  1.11s/it]


L = 0.9, np.mean(results) = 0.5619801980198019


100%|██████████| 101/101 [02:04<00:00,  1.24s/it]

L = 1.0, np.mean(results) = 0.5293069306930693


In [18]:
from sklearn.linear_model import Ridge

for L in np.linspace(0, 1, 11):
    GE = ma.get_game_embs().numpy()

    results = list()
    K = 1
    T = 100
    R_All = ctx.get_relevs("key")

    def get_Rp(GE, A, R):
        Ae = GE[A]

        c = Ridge(fit_intercept=True)
        c.fit(Ae, R)

        Rp = c.predict(GE)


        return Rp

    As = list()
    for i in tqdm.tqdm(range(R_All.shape[0])):
        A = ctx.key_games[:K]
        Rt = R_All[i]
        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

        for _ in range(T // K - 1):
            Rp = get_Rp(GE, A, Rt[A])

            An = list(A)
            for ai in Rp.argsort()[::-1]:
                if ai not in An:
                    An.append(ai)
                if len(An) == len(A) + K:
                    break
            A = np.array(An)
        assert len(A) == T
        
        Rp1 = get_Rp(GE, A, Rt[A])
        Rp2 = GE @ Rt[ma.key_cols_idx]
        
        Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
        Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()
        
        Rp = L * Rp1 + (1. - L) * Rp2

        As.append(A)

        pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
        results.append(ev(pred_top, gt_top) / float(100))

    print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 101/101 [01:38<00:00,  1.03it/s]


L = 0.0, np.mean(results) = 0.7029702970297029


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 101/101 [01:21<00:00,  1.23it/s]


L = 0.1, np.mean(results) = 0.7087128712871286


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 101/101 [01:11<00:00,  1.40it/s]


...

L = 0.8, np.mean(results) = 0.5780198019801981


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 101/101 [01:10<00:00,  1.42it/s]


L = 0.9, np.mean(results) = 0.5585148514851486


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 101/101 [01:23<00:00,  1.22it/s]

L = 1.0, np.mean(results) = 0.54009900990099


In [10]:
L = 0.1
GE = ma.get_game_embs().numpy()

results = list()
K = 5
T = 100
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(fit_intercept=True)
    c.fit(Ae, R)

    Rp = c.predict(GE)


    return Rp

As = list()
for i in tqdm.tqdm(range(R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        Rp = get_Rp(GE, A, Rt[A])

        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
    assert len(A) == T

    Rp1 = get_Rp(GE, A, Rt[A])
    Rp2 = GE @ Rt[ma.key_cols_idx]

    Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
    Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()

    Rp = L * Rp1 + (1. - L) * Rp2

    As.append(A)

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2034/2034 [06:33<00:00,  5.17it/s]

L = 0.1, np.mean(results) = 0.6992772861356932


In [19]:
L = 0.1
GE = ma.get_game_embs().numpy()

results = list()
K = 1
T = 100
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(fit_intercept=True)
    c.fit(Ae, R)

    Rp = c.predict(GE)


    return Rp

As = list()
for i in tqdm.tqdm(range(R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        Rp = get_Rp(GE, A, Rt[A])

        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
    assert len(A) == T

    Rp1 = get_Rp(GE, A, Rt[A])
    Rp2 = GE @ Rt[ma.key_cols_idx]

    Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
    Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()

    Rp = L * Rp1 + (1. - L) * Rp2

    As.append(A)

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2034/2034 [25:14<00:00,  1.34it/s]

L = 0.1, np.mean(results) = 0.7342182890855458


In [13]:
t = ctx.get_relevs("train").T
t = (t - t.mean()) / t.std()  # like in other comparisons with RBE
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("key"), ma.get_score("train"), ma.get_score("test")

/tmp/ipykernel_924375/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.7005940594059407 0.36494004299742316 101
np.mean(results), mse, len(results) =  0.7277067183462532 0.17279571845624656 4644
np.mean(results), mse, len(results) =  0.7196509341199606 0.270004741409769 2034


(0.7005940594059407, 0.7277067183462532, 0.7196509341199606)

In [14]:
from sklearn.linear_model import Ridge

for L in np.linspace(0, 1, 11):
    GE = ma.get_game_embs().numpy()

    results = list()
    K = 5
    T = 100
    R_All = ctx.get_relevs("key")

    def get_Rp(GE, A, R):
        Ae = GE[A]

        c = Ridge(fit_intercept=True)
        c.fit(Ae, R)

        Rp = c.predict(GE)


        return Rp

    As = list()
    for i in tqdm.tqdm(range(R_All.shape[0])):
        A = ctx.key_games[:K]
        Rt = R_All[i]
        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

        for _ in range(T // K - 1):
            Rp = get_Rp(GE, A, Rt[A])

            An = list(A)
            for ai in Rp.argsort()[::-1]:
                if ai not in An:
                    An.append(ai)
                if len(An) == len(A) + K:
                    break
            A = np.array(An)
        assert len(A) == T
        
        Rp1 = get_Rp(GE, A, Rt[A])
        Rp2 = GE @ Rt[ma.key_cols_idx]
        
        Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
        Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()
        
        Rp = L * Rp1 + (1. - L) * Rp2

        As.append(A)

        pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
        results.append(ev(pred_top, gt_top) / float(100))

    print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 101/101 [00:26<00:00,  3.78it/s]


L = 0.0, np.mean(results) = 0.7005940594059407


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 101/101 [00:17<00:00,  5.65it/s]


L = 0.1, np.mean(results) = 0.6942574257425742


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 101/101 [00:21<00:00,  4.63it/s]


...

L = 0.8, np.mean(results) = 0.5813861386138615


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 101/101 [00:16<00:00,  6.17it/s]


L = 0.9, np.mean(results) = 0.560891089108911


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 101/101 [00:19<00:00,  5.15it/s]

L = 1.0, np.mean(results) = 0.5251485148514851


In [15]:
L = 0.1
GE = ma.get_game_embs().numpy()

results = list()
K = 5
T = 100
R_All = ctx.get_relevs("test")

def get_Rp(GE, A, R):
    Ae = GE[A]

    c = Ridge(fit_intercept=True)
    c.fit(Ae, R)

    Rp = c.predict(GE)


    return Rp

As = list()
for i in tqdm.tqdm(range(R_All.shape[0])):
    A = ctx.key_games[:K]
    Rt = R_All[i]
    ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

    for _ in range(T // K - 1):
        Rp = get_Rp(GE, A, Rt[A])

        An = list(A)
        for ai in Rp.argsort()[::-1]:
            if ai not in An:
                An.append(ai)
            if len(An) == len(A) + K:
                break
        A = np.array(An)
    assert len(A) == T

    Rp1 = get_Rp(GE, A, Rt[A])
    Rp2 = GE @ Rt[ma.key_cols_idx]

    Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
    Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()

    Rp = L * Rp1 + (1. - L) * Rp2

    As.append(A)

    pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
    results.append(ev(pred_top, gt_top) / float(100))

print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2034/2034 [05:19<00:00,  6.36it/s]

L = 0.1, np.mean(results) = 0.7158751229105212


In [16]:
from sklearn.linear_model import Ridge

for L in np.linspace(0, 1, 11):
    GE = ma.get_game_embs().numpy()

    results = list()
    K = 1
    T = 100
    R_All = ctx.get_relevs("key")

    def get_Rp(GE, A, R):
        Ae = GE[A]

        c = Ridge(fit_intercept=True)
        c.fit(Ae, R)

        Rp = c.predict(GE)


        return Rp

    As = list()
    for i in tqdm.tqdm(range(R_All.shape[0])):
        A = ctx.key_games[:K]
        Rt = R_All[i]
        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

        for _ in range(T // K - 1):
            Rp = get_Rp(GE, A, Rt[A])

            An = list(A)
            for ai in Rp.argsort()[::-1]:
                if ai not in An:
                    An.append(ai)
                if len(An) == len(A) + K:
                    break
            A = np.array(An)
        assert len(A) == T
        
        Rp1 = get_Rp(GE, A, Rt[A])
        Rp2 = GE @ Rt[ma.key_cols_idx]
        
        Rp1 = (Rp1 - Rp1.mean()) / Rp1.std()
        Rp2 = (Rp2 - Rp2.mean()) / Rp2.std()
        
        Rp = L * Rp1 + (1. - L) * Rp2

        As.append(A)

        pred_top, gt_top = Rp.argsort()[::-1][:100], Rt.argsort()[::-1][:100]
        results.append(ev(pred_top, gt_top) / float(100))

    print(f"L = {L}, np.mean(results) = {np.mean(results)}")

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 101/101 [01:49<00:00,  1.08s/it]


L = 0.0, np.mean(results) = 0.7005940594059407


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 101/101 [01:37<00:00,  1.03it/s]


L = 0.1, np.mean(results) = 0.6888118811881188


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 101/101 [01:13<00:00,  1.37it/s]


...

L = 0.8, np.mean(results) = 0.5606930693069306


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 101/101 [01:11<00:00,  1.41it/s]


L = 0.9, np.mean(results) = 0.5453465346534653


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 101/101 [01:16<00:00,  1.32it/s]

L = 1.0, np.mean(results) = 0.5232673267326732
